# ⭐ Day 37: Correlation Analysis & Heatmaps
## Key Insights for Feature Selection in AI & ML

**Day 37 of 369-day Python & AI Learning Path** 🚀


## Welcome to Day 37!

Welcome back to our Python & AI Learning Path! Today we are diving into one of the most fundamental yet powerful techniques in data science: **correlation analysis and heatmap visualization**. Whether you are building your first regression model or optimizing a production pipeline, understanding how features relate to each other is absolutely critical for success.

Correlation analysis serves as the bridge between raw data and actionable insights. It reveals which features move together, which are redundant, and most importantly, which features have the strongest predictive power for your target variable. In machine learning, this knowledge directly translates to better feature selection, reduced overfitting, and more interpretable models. A well-crafted correlation heatmap can communicate more insights in a single glance than hours of tabular exploration.

Beyond aesthetics, correlation analysis has practical modeling implications. High correlation between features (multicollinearity) can destabilize linear models and inflate standard errors. Understanding these relationships helps you decide which features to keep, combine, or drop. Similarly, analyzing target correlations guides feature engineering efforts and helps set realistic performance expectations.

In this notebook, we will explore Pearson, Spearman, and Kendall correlations, create publication-quality heatmaps, detect multicollinearity, and translate correlation insights into modeling decisions. By the end, you will have a comprehensive toolkit for feature relationship analysis that you can apply to any ML project. Let us uncover the hidden connections in your data! 🔗


## 📋 Table of Contents

1. [What is Correlation and Why It Matters in ML](#1-what-is-correlation-and-why-it-matters-in-ml)
2. [Types of Correlation](#2-types-of-correlation)
3. [Loading Dataset and Preparing Numerical Features](#3-loading-dataset-and-preparing-numerical-features)
4. [Computing Correlation Matrix](#4-computing-correlation-matrix)
5. [Visualizing Correlation with Heatmaps](#5-visualizing-correlation-with-heatmaps)
6. [Interpreting Correlation Values](#6-interpreting-correlation-values)
7. [Correlation with Target Variable](#7-correlation-with-target-variable)
8. [Detecting Multicollinearity](#8-detecting-multicollinearity)
9. [Feature Selection Using Correlation](#9-feature-selection-using-correlation)
10. [Limitations of Correlation Analysis](#10-limitations-of-correlation-analysis)
11. [🛠️ Hands-On Exercises](#-hands-on-exercises)
12. [Solutions & Key Insights](#-solutions--key-insights)


## 1. What is Correlation and Why It Matters in ML 🔗

### Definition
Correlation measures the statistical relationship between two variables, ranging from -1 (perfect negative relationship) to +1 (perfect positive relationship). A value of 0 indicates no linear relationship.

### Why Correlation Analysis is Critical in ML:

- **🎯 Feature Selection**: Identify features most predictive of the target
- **🔄 Redundancy Detection**: Find highly correlated features that provide duplicate information
- **⚠️ Multicollinearity**: Detect feature pairs that can destabilize linear models
- **💡 Domain Insights**: Understand underlying data generating processes
- **📊 EDA Efficiency**: Quickly identify promising features for modeling

### Modeling Implications:

| Correlation Insight | Modeling Action |
|---------------------|-----------------|
| Feature-Target: High positive | Strong predictor, prioritize |
| Feature-Target: Near zero | Weak predictor, consider dropping |
| Feature-Feature: High (>0.8) | Multicollinearity risk, drop one |
| Feature-Feature: Moderate (0.5-0.8) | Potential interaction term |

**💡 Key Principle**: Correlation is not causation, but it is a powerful signal for prediction!


## 2. Types of Correlation 📊

Different correlation metrics capture different types of relationships:

### 🔹 Pearson Correlation (r)
**Best for**: Linear relationships between continuous variables
- Measures linear correlation
- Sensitive to outliers
- Range: [-1, 1]
- Assumes normal distribution

**Formula**: r = cov(X,Y) / (σ_X * σ_Y)

### 🔹 Spearman Correlation (ρ)
**Best for**: Monotonic relationships, ordinal data, non-normal distributions
- Rank-based correlation
- Robust to outliers
- Captures any monotonic pattern (not just linear)
- Range: [-1, 1]

### 🔹 Kendall's Tau (τ)
**Best for**: Small samples, many tied ranks
- Also rank-based
- More robust than Spearman for small samples
- Computationally intensive for large datasets
- Range: [-1, 1]

### 🔹 Point-Biserial
**Best for**: One continuous, one binary variable
- Special case of Pearson
- Useful for binary classification targets

**📋 Selection Guide**:
- Use **Pearson** for linear, normal, continuous data
- Use **Spearman** for skewed data, ordinal features, or unknown relationships
- Use **Kendall** for small samples with many ties
- Compare Pearson vs Spearman to detect non-linear relationships!


In [ ]:
# Import essential libraries for correlation analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr, kendalltau
import warnings
warnings.filterwarnings('ignore')

# Set visualization parameters
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)
np.random.seed(42)

print('✅ Libraries loaded successfully!')
print(f'NumPy version: {np.__version__}')
print(f'Pandas version: {pd.__version__}')
print(f'Seaborn version: {sns.__version__}')


In [ ]:
# Create a rich synthetic dataset with clear correlation structures
# Simulating a comprehensive real estate / housing dataset

n_samples = 1000

# Create base features with realistic distributions and relationships

# Size-related features (highly correlated cluster)
square_feet = np.random.lognormal(7.5, 0.5, n_samples)
lot_size = square_feet * 2.5 + np.random.normal(0, 500, n_samples)  # Correlated with sqft
living_area = square_feet * 0.85 + np.random.normal(0, 100, n_samples)  # Highly correlated

# Room counts (correlated with size)
bedrooms = (square_feet / 400 + np.random.normal(0, 0.5, n_samples)).clip(1, 6).round().astype(int)
bathrooms = (bedrooms * 0.7 + np.random.normal(0, 0.3, n_samples)).clip(1, 4).round().astype(int)
total_rooms = bedrooms + bathrooms + np.random.poisson(2, n_samples)

# Quality and location features
school_rating = np.random.beta(7, 2, n_samples) * 10  # 0-10 scale
distance_to_city = np.random.exponential(15, n_samples).clip(1, 50)
crime_rate = np.random.exponential(0.05, n_samples).clip(0, 0.5)
walkability = np.random.beta(5, 5, n_samples) * 100  # 0-100 scale

# Property characteristics
age_years = np.random.exponential(20, n_samples).clip(0, 100)
garage_spaces = np.random.poisson(1.5, n_samples).clip(0, 3).astype(int)
num_amenities = np.random.poisson(4, n_samples)
renovation_quality = np.random.beta(3, 2, n_samples) * 10  # 0-10

# Create target variable (price) with realistic relationships
base_price = (
    square_feet * 120 +
    lot_size * 15 +
    bedrooms * 15000 +
    bathrooms * 25000 +
    school_rating * 8000 +
    renovation_quality * 5000 -
    age_years * 800 -
    distance_to_city * 2000 -
    crime_rate * 400000 +
    walkability * 300 +
    np.random.normal(0, 30000, n_samples)  # noise
).clip(80000, 1500000)

# Add non-linear relationship (price per sqft decreases with size - economies of scale)
price_per_sqft_premium = 50 * np.exp(-square_feet / 3000)
price = base_price + price_per_sqft_premium * square_feet

# Assemble dataframe
df = pd.DataFrame({
    'square_feet': square_feet,
    'lot_size': lot_size,
    'living_area': living_area,
    'bedrooms': bedrooms,
    'bathrooms': bathrooms,
    'total_rooms': total_rooms,
    'age_years': age_years,
    'school_rating': school_rating,
    'distance_to_city': distance_to_city,
    'crime_rate': crime_rate,
    'walkability': walkability,
    'garage_spaces': garage_spaces,
    'num_amenities': num_amenities,
    'renovation_quality': renovation_quality,
    'price': price
})

print('✅ Rich synthetic dataset created with intentional correlation structures!')
print(f'Dataset shape: {df.shape}')
print(f'Features: {len(df.columns) - 1} (excluding target price)')
print(f'\nDataset Info:')
print(df.info())


## 3. Loading Dataset and Preparing Numerical Features 📋

Let us examine our dataset and prepare it for correlation analysis.


In [ ]:
# Comprehensive dataset overview
print('=' * 70)
print('📊 DATASET OVERVIEW')
print('=' * 70)
print(f'Total samples: {len(df):,}')
print(f'Total features: {len(df.columns)}')
print(f'Target variable: price')
print(f'\nFirst 5 rows:')
print(df.head())

print(f'\nBasic Statistics:')
print(df.describe().round(2))

# Check for missing values
print(f'\nMissing Values:')
missing = df.isnull().sum()
if missing.sum() == 0:
    print('✅ No missing values detected')
else:
    print(missing[missing > 0])


## 4. Computing Correlation Matrix 🧮

Now let us compute correlations using different methods and compare them.


In [ ]:
# Compute correlation matrices using different methods
print('=' * 70)
print('🔗 COMPUTING CORRELATION MATRICES')
print('=' * 70)

# Pearson correlation (default)
pearson_corr = df.corr(method='pearson')

# Spearman correlation (rank-based)
spearman_corr = df.corr(method='spearman')

# Kendall correlation
kendall_corr = df.corr(method='kendall')

print('✅ Correlation matrices computed:')
print(f'  • Pearson (linear):   {pearson_corr.shape}')
print(f'  • Spearman (rank):    {spearman_corr.shape}')
print(f'  • Kendall (concordance): {kendall_corr.shape}')

# Display correlation with target for all methods
print(f'\n📊 Correlation with Target (Price) - Comparison:')
print('-' * 70)
target_corr_comparison = pd.DataFrame({
    'Feature': pearson_corr.index[:-1],  # Exclude price itself
    'Pearson': pearson_corr['price'].values[:-1],
    'Spearman': spearman_corr['price'].values[:-1],
    'Kendall': kendall_corr['price'].values[:-1]
})
target_corr_comparison = target_corr_comparison.sort_values('Pearson', key=abs, ascending=False)
print(target_corr_comparison.round(3).to_string(index=False))


## 5. Visualizing Correlation with Heatmaps 🔥

Heatmaps are the most effective way to visualize correlation matrices. Let us create multiple styles.


In [ ]:
# Create a comprehensive correlation heatmap dashboard
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# 1. Basic heatmap with annotations
ax1 = axes[0, 0]
mask = np.triu(np.ones_like(pearson_corr, dtype=bool), k=1)  # Mask upper triangle
sns.heatmap(pearson_corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', 
            center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8}, 
            ax=ax1, vmin=-1, vmax=1)
ax1.set_title('🔥 Basic Correlation Heatmap (Pearson)\nLower Triangle with Annotations', 
              fontsize=11, fontweight='bold')

# 2. Full heatmap with strong correlations highlighted
ax2 = axes[0, 1]
sns.heatmap(pearson_corr, annot=False, cmap='RdBu_r', center=0, 
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8}, 
            ax=ax2, vmin=-1, vmax=1)
ax2.set_title('🎨 Full Correlation Matrix\n(Color intensity = correlation strength)', 
              fontsize=11, fontweight='bold')

# 3. Spearman vs Pearson comparison (absolute difference)
ax3 = axes[1, 0]
diff_corr = np.abs(pearson_corr - spearman_corr)
sns.heatmap(diff_corr, annot=True, fmt='.2f', cmap='YlOrRd', 
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8, 'label': '|Difference|'}, 
            ax=ax3, vmin=0, vmax=1)
ax3.set_title('📊 Pearson vs Spearman Difference\n(High values = non-linear relationships)', 
              fontsize=11, fontweight='bold')

# 4. Target correlation focused heatmap
ax4 = axes[1, 1]
target_corr = pearson_corr[['price']].sort_values('price', ascending=False)
sns.heatmap(target_corr, annot=True, fmt='.3f', cmap='RdBu_r', 
            center=0, linewidths=0.5, cbar_kws={'shrink': 0.8}, 
            ax=ax4, vmin=-1, vmax=1)
ax4.set_title('🎯 Target Correlations Only\n(Feature importance ranking)', 
              fontsize=11, fontweight='bold')
ax4.set_xlabel('')

plt.suptitle('📈 Correlation Heatmap Dashboard\nMultiple Perspectives on Feature Relationships', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('💡 Insight: The difference heatmap reveals which relationships are non-linear (Spearman ≠ Pearson)!')


In [ ]:
# Create a publication-quality styled heatmap
fig, ax = plt.subplots(figsize=(12, 10))

# Create custom colormap
cmap = sns.diverging_palette(250, 10, as_cmap=True, center='light')

# Mask upper triangle for cleaner look
mask = np.triu(np.ones_like(pearson_corr, dtype=bool), k=1)

# Create heatmap with custom styling
heatmap = sns.heatmap(pearson_corr, mask=mask, annot=True, fmt='.2f', 
                      cmap=cmap, center=0, square=True, linewidths=1,
                      linecolor='white', cbar_kws={'shrink': 0.8, 'label': 'Correlation Coefficient'},
                      annot_kws={'size': 10, 'weight': 'bold'}, ax=ax,
                      vmin=-1, vmax=1)

# Styling
ax.set_title('🔗 Feature Correlation Matrix\nPearson Correlation Coefficients', 
             fontsize=14, fontweight='bold', pad=20)

# Rotate labels for better readability
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

# Add border
for _, spine in heatmap.spines.items():
    spine.set_visible(True)
    spine.set_linewidth(1.5)

plt.tight_layout()
plt.show()

print('✅ Publication-quality heatmap created!')


## 6. Interpreting Correlation Values 💡

Understanding what correlation values mean in practice:

### 📊 Correlation Strength Guidelines

| Absolute Value | Strength | Interpretation |
|----------------|----------|----------------|
| 0.00 - 0.19 | Very Weak | Negligible relationship |
| 0.20 - 0.39 | Weak | Small but noticeable |
| 0.40 - 0.59 | Moderate | Clear relationship |
| 0.60 - 0.79 | Strong | Important relationship |
| 0.80 - 1.00 | Very Strong | Redundant or near-duplicate |

### 🔗 Direction Matters
- **Positive (+)**: As one variable increases, the other increases
- **Negative (-)**: As one variable increases, the other decreases
- **Zero (~0)**: No linear relationship (but may have non-linear!)

### ⚠️ Critical Interpretation Rules
1. **Context is king**: 0.3 may be strong in social science, weak in physics
2. **Non-linear blindspot**: Correlation ≠ 0 does not mean no relationship
3. **Outlier sensitivity**: A single extreme point can inflate correlation
4. **Sample size matters**: Small samples produce unstable correlations


In [ ]:
# Detailed correlation interpretation analysis
print('=' * 70)
print('💡 DETAILED CORRELATION INTERPRETATION')
print('=' * 70)

def interpret_correlation(value):
    """Provide interpretation of correlation strength"""
    abs_val = abs(value)
    direction = 'positive' if value > 0 else 'negative'
    
    if abs_val >= 0.8:
        strength = 'VERY STRONG'
        action = '⚠️ High multicollinearity risk - consider dropping one'
    elif abs_val >= 0.6:
        strength = 'STRONG'
        action = '✅ Important relationship - key feature'
    elif abs_val >= 0.4:
        strength = 'MODERATE'
        action = '📊 Useful relationship - include in model'
    elif abs_val >= 0.2:
        strength = 'WEAK'
        action = '🤔 Marginal value - test in feature selection'
    else:
        strength = 'VERY WEAK/NEGLIGIBLE'
        action = '❌ Likely not predictive - candidate for removal'
    
    return strength, direction, action

# Analyze all feature pairs
print('\n📋 Top 10 Strongest Feature Correlations (excluding target):')
print('-' * 70)

# Get upper triangle correlations (avoid duplicates)
upper_triangle = pearson_corr.where(np.triu(np.ones(pearson_corr.shape), k=1).astype(bool))
correlation_pairs = []

for col in upper_triangle.columns:
    for idx in upper_triangle.index:
        value = upper_triangle.loc[idx, col]
        if not pd.isna(value) and idx != 'price' and col != 'price':
            correlation_pairs.append((idx, col, value))

# Sort by absolute correlation
correlation_pairs.sort(key=lambda x: abs(x[2]), reverse=True)

for feat1, feat2, corr in correlation_pairs[:10]:
    strength, direction, action = interpret_correlation(corr)
    print(f'{feat1} ↔ {feat2}: {corr:>6.3f} ({strength} {direction})')
    print(f'   {action}')
    print()


## 7. Correlation with Target Variable 🎯

The most critical analysis: which features predict our target?


In [ ]:
# Comprehensive target correlation analysis
print('=' * 70)
print('🎯 TARGET VARIABLE CORRELATION ANALYSIS')
print('=' * 70)

target_correlations = pearson_corr['price'].drop('price').sort_values(key=abs, ascending=False)

print('\n📊 All Features Ranked by Correlation with Price:')
print('-' * 70)

for feature, corr in target_correlations.items():
    strength, direction, action = interpret_correlation(corr)
    bar = '█' * int(abs(corr) * 20)  # Visual bar
    print(f'{feature:20s}: {corr:>6.3f} {bar:<20} {strength}')

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 1. Horizontal bar chart
ax1 = axes[0]
colors = ['green' if x > 0 else 'red' for x in target_correlations.values]
bars = ax1.barh(target_correlations.index, target_correlations.values, color=colors, alpha=0.7, edgecolor='black')
ax1.set_xlabel('Correlation with Price', fontsize=11)
ax1.set_title('📊 Feature Importance: Correlation with Target\n(Green=Positive, Red=Negative)', 
              fontsize=12, fontweight='bold')
ax1.axvline(x=0, color='black', linewidth=0.8)
ax1.grid(axis='x', alpha=0.3)

# Add value labels
for i, (idx, val) in enumerate(target_correlations.items()):
    ax1.text(val + 0.01 if val > 0 else val - 0.01, i, f'{val:.3f}', 
             va='center', ha='left' if val > 0 else 'right', fontsize=9)

# 2. Absolute correlation ranking
ax2 = axes[1]
abs_corr = target_correlations.abs().sort_values(ascending=True)
colors_abs = plt.cm.RdYlGn(abs_corr.values)
bars2 = ax2.barh(abs_corr.index, abs_corr.values, color=colors_abs, alpha=0.8, edgecolor='black')
ax2.set_xlabel('Absolute Correlation |r|', fontsize=11)
ax2.set_title('🔥 Predictive Power Ranking\n(Absolute correlation = importance)', 
              fontsize=12, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\n💡 Key Insights:')
print(f'   • Top predictor: {target_correlations.abs().idxmax()} (r={target_correlations.abs().max():.3f})')
print(f'   • Features with |r| > 0.5: {(target_correlations.abs() > 0.5).sum()}')
print(f'   • Negative predictors: {(target_correlations < 0).sum()}')


In [ ]:
# Scatter plots for top correlated features
top_features = target_correlations.abs().head(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for idx, feature in enumerate(top_features):
    ax = axes[idx // 2, idx % 2]
    
    # Scatter plot with regression line
    ax.scatter(df[feature], df['price'], alpha=0.5, c='steelblue', edgecolors='black', linewidth=0.5, s=50)
    
    # Add regression line
    z = np.polyfit(df[feature], df['price'], 1)
    p = np.poly1d(z)
    ax.plot(df[feature].sort_values(), p(df[feature].sort_values()), 'r--', alpha=0.8, linewidth=2, label='Trend')
    
    # Calculate and display R-squared
    r_squared = pearson_corr.loc[feature, 'price'] ** 2
    
    ax.set_xlabel(feature.replace('_', ' ').title(), fontsize=11)
    ax.set_ylabel('Price ($)', fontsize=11)
    ax.set_title(f'{feature.replace("_", " ").title()} vs Price\nr = {pearson_corr.loc[feature, "price"]:.3f}, R² = {r_squared:.3f}', 
                 fontsize=11, fontweight='bold')
    ax.grid(alpha=0.3)
    ax.legend()

plt.suptitle('📈 Scatter Plots: Top Correlated Features with Target\n(R² = variance explained)', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('💡 Insight: R² shows how much price variance is explained by each feature individually!')


## 8. Detecting Multicollinearity ⚠️

Multicollinearity occurs when features are highly correlated with each other, causing problems for linear models.


In [ ]:
# Multicollinearity detection and analysis
print('=' * 70)
print('⚠️ MULTICOLLINEARITY DETECTION')
print('=' * 70)

# Define thresholds
HIGH_CORR_THRESHOLD = 0.8
MODERATE_CORR_THRESHOLD = 0.6

# Find high correlations (excluding target)
feature_corr = pearson_corr.drop('price', axis=0).drop('price', axis=1)

high_corr_pairs = []
moderate_corr_pairs = []

for i in range(len(feature_corr.columns)):
    for j in range(i+1, len(feature_corr.columns)):
        corr_val = feature_corr.iloc[i, j]
        feat1 = feature_corr.columns[i]
        feat2 = feature_corr.columns[j]
        
        if abs(corr_val) >= HIGH_CORR_THRESHOLD:
            high_corr_pairs.append((feat1, feat2, corr_val))
        elif abs(corr_val) >= MODERATE_CORR_THRESHOLD:
            moderate_corr_pairs.append((feat1, feat2, corr_val))

print(f'\n🔴 HIGH Multicollinearity (|r| ≥ {HIGH_CORR_THRESHOLD}):')
if high_corr_pairs:
    for feat1, feat2, corr in sorted(high_corr_pairs, key=lambda x: abs(x[2]), reverse=True):
        print(f'   {feat1} ↔ {feat2}: {corr:.3f} ⚠️ CRITICAL')
else:
    print('   ✅ No critical multicollinearity detected')

print(f'\n🟡 MODERATE Multicollinearity ({MODERATE_CORR_THRESHOLD} ≤ |r| < {HIGH_CORR_THRESHOLD}):')
if moderate_corr_pairs:
    for feat1, feat2, corr in sorted(moderate_corr_pairs, key=lambda x: abs(x[2]), reverse=True):
        print(f'   {feat1} ↔ {feat2}: {corr:.3f}')
else:
    print('   ✅ No moderate multicollinearity detected')

# Variance Inflation Factor (VIF) calculation
from statsmodels.stats.outliers_influence import variance_inflation_factor

print(f'\n📊 Variance Inflation Factor (VIF) Analysis:')
print('-' * 50)
print('VIF > 10: High multicollinearity')
print('VIF > 5:  Moderate multicollinearity')
print('-' * 50)

# Prepare data for VIF (handle any issues)
X_vif = df.drop('price', axis=1).copy()
X_vif = X_vif.fillna(X_vif.median())
X_vif = X_vif.clip(lower=0)  # Ensure positive for some features

vif_data = pd.DataFrame()
vif_data['Feature'] = X_vif.columns
vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
vif_data = vif_data.sort_values('VIF', ascending=False)

print(vif_data.round(2).to_string(index=False))


In [ ]:
# Visualize multicollinearity
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 1. Correlation matrix focused on high correlations
ax1 = axes[0]
high_corr_matrix = feature_corr.copy()

# Mask correlations below threshold for cleaner view
mask_low = np.abs(high_corr_matrix) < 0.5
sns.heatmap(high_corr_matrix, mask=mask_low, annot=True, fmt='.2f', 
            cmap='RdBu_r', center=0, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8, 'label': 'Correlation'}, ax=ax1,
            vmin=-1, vmax=1)
ax1.set_title('⚠️ Multicollinearity Heatmap\n(Only |r| ≥ 0.5 shown)', 
              fontsize=12, fontweight='bold')

# 2. VIF bar chart
ax2 = axes[1]
colors = ['red' if x > 10 else 'orange' if x > 5 else 'green' for x in vif_data['VIF']]
bars = ax2.barh(vif_data['Feature'], vif_data['VIF'], color=colors, alpha=0.7, edgecolor='black')
ax2.axvline(x=5, color='orange', linestyle='--', linewidth=2, label='VIF = 5')
ax2.axvline(x=10, color='red', linestyle='--', linewidth=2, label='VIF = 10')
ax2.set_xlabel('Variance Inflation Factor (VIF)', fontsize=11)
ax2.set_title('📊 VIF by Feature\n(Green=OK, Orange=Caution, Red=Critical)', 
              fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(axis='x', alpha=0.3)

# Add value labels
for i, (idx, row) in enumerate(vif_data.iterrows()):
    ax2.text(row['VIF'] + 0.5, i, f'{row["VIF"]:.1f}', 
             va='center', fontsize=9)

plt.tight_layout()
plt.show()

print('💡 Insight: Features with VIF > 10 are highly redundant—consider removing or combining them!')


## 9. Feature Selection Using Correlation 🎯

Translate correlation insights into actionable feature selection decisions.


In [ ]:
# Feature selection strategy based on correlation analysis
print('=' * 70)
print('🎯 FEATURE SELECTION STRATEGY')
print('=' * 70)

# Define selection criteria
TARGET_CORR_THRESHOLD = 0.3  # Minimum correlation with target
MULTICOLLINEARITY_THRESHOLD = 0.85  # Maximum correlation between features

# Step 1: Filter by target correlation
strong_target_corr = target_correlations.abs()[target_correlations.abs() >= TARGET_CORR_THRESHOLD]
weak_target_corr = target_correlations.abs()[target_correlations.abs() < TARGET_CORR_THRESHOLD]

print(f'\n📊 Step 1: Target Correlation Filter (|r| ≥ {TARGET_CORR_THRESHOLD})')
print(f'   ✅ KEEP ({len(strong_target_corr)} features): {list(strong_target_corr.index)}')
print(f'   ❌ DROP ({len(weak_target_corr)} features): {list(weak_target_corr.index)}')

# Step 2: Remove multicollinearity from kept features
features_to_check = list(strong_target_corr.index)
final_features = []
dropped_for_multicollinearity = []

# Sort by target correlation (descending) to keep most predictive
features_sorted = target_correlations[features_to_check].abs().sort_values(ascending=False).index.tolist()

for feature in features_sorted:
    # Check correlation with already selected features
    is_redundant = False
    for selected in final_features:
        corr_between = pearson_corr.loc[feature, selected]
        if abs(corr_between) > MULTICOLLINEARITY_THRESHOLD:
            is_redundant = True
            dropped_for_multicollinearity.append((feature, selected, corr_between))
            break
    
    if not is_redundant:
        final_features.append(feature)

print(f'\n📊 Step 2: Multicollinearity Removal (|r| < {MULTICOLLINEARITY_THRESHOLD} between features)')
print(f'   ✅ FINAL FEATURES ({len(final_features)}): {final_features}')
if dropped_for_multicollinearity:
    print(f'   ⚠️ Dropped due to multicollinearity:')
    for feat, conflict, corr in dropped_for_multicollinearity:
        print(f'      • {feat} (conflicts with {conflict}, r={corr:.3f})')

# Summary statistics
print(f'\n📈 SELECTION SUMMARY:')
print(f'   Original features: {len(df.columns) - 1}')
print(f'   After target filter: {len(strong_target_corr)}')
print(f'   Final features: {len(final_features)}')
print(f'   Reduction: {(1 - len(final_features)/(len(df.columns)-1))*100:.1f}%')


In [ ]:
# Visualize feature selection process
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Target correlation threshold visualization
ax1 = axes[0, 0]
all_target_corrs = target_correlations.abs().sort_values(ascending=True)
colors = ['red' if x < TARGET_CORR_THRESHOLD else 'green' for x in all_target_corrs.values]
bars1 = ax1.barh(all_target_corrs.index, all_target_corrs.values, color=colors, alpha=0.7, edgecolor='black')
ax1.axvline(x=TARGET_CORR_THRESHOLD, color='red', linestyle='--', linewidth=2, 
            label=f'Threshold = {TARGET_CORR_THRESHOLD}')
ax1.set_xlabel('|Correlation with Target|', fontsize=10)
ax1.set_title('📊 Step 1: Target Correlation Filter', fontsize=11, fontweight='bold')
ax1.legend()
ax1.grid(axis='x', alpha=0.3)

# 2. Selected features final ranking
ax2 = axes[0, 1]
final_target_corrs = target_correlations[final_features].abs().sort_values(ascending=True)
bars2 = ax2.barh(final_target_corrs.index, final_target_corrs.values, 
                  color='forestgreen', alpha=0.8, edgecolor='black')
ax2.set_xlabel('|Correlation with Target|', fontsize=10)
ax2.set_title(f'✅ Step 2: Final Selected Features (n={len(final_features)})', 
              fontsize=11, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

# 3. Feature correlation network (simplified)
ax3 = axes[1, 0]
final_corr_matrix = pearson_corr.loc[final_features, final_features]
sns.heatmap(final_corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', 
            center=0, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8}, ax=ax3, vmin=-1, vmax=1)
ax3.set_title('🔗 Final Feature Correlation Matrix\n(Low multicollinearity)', 
              fontsize=11, fontweight='bold')

# 4. Selection flow diagram (as bar chart)
ax4 = axes[1, 1]
stages = ['Original', 'After Target\nFilter', 'After Multicollinearity\nRemoval']
counts = [len(df.columns)-1, len(strong_target_corr), len(final_features)]
colors_flow = ['steelblue', 'orange', 'forestgreen']
bars4 = ax4.bar(stages, counts, color=colors_flow, alpha=0.8, edgecolor='black', width=0.6)
ax4.set_ylabel('Number of Features', fontsize=10)
ax4.set_title('📉 Feature Selection Flow', fontsize=11, fontweight='bold')
ax4.grid(axis='y', alpha=0.3)

# Add value labels
for bar in bars4:
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height + 0.2,
             f'{int(height)}', ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.suptitle('🎯 Feature Selection Pipeline Based on Correlation Analysis', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('✅ Feature selection visualization complete!')


## 10. Limitations of Correlation Analysis ⚠️

Understanding what correlation cannot tell you is as important as understanding what it can.

### 🔴 Critical Limitations

1. **Causation ≠ Correlation**
   - Ice cream sales correlate with drowning incidents (both depend on temperature)
   - Always ask: Is there a confounding variable?

2. **Non-Linear Relationships**
   - Correlation measures *linear* relationships only
   - X and X² have zero correlation despite perfect relationship
   - **Solution**: Compare Pearson vs Spearman, use scatter plots

3. **Outlier Sensitivity**
   - A single extreme point can create spurious correlation
   - **Solution**: Use Spearman/Kendall for robustness

4. **Simpson's Paradox**
   - Trend reverses when data is aggregated
   - Correlation in subgroups may differ from overall

5. **Range Restriction**
   - Correlation underestimates true relationship if data range is limited

### ✅ Best Practices to Mitigate Limitations

- **Always visualize**: Scatter plots reveal non-linear patterns
- **Check robustness**: Compare Pearson, Spearman, and Kendall
- **Control for confounders**: Use partial correlation when needed
- **Beware small samples**: Correlations are unstable with n < 30
- **Consider interactions**: High correlation with target may indicate interaction potential

**💡 Key Takeaway**: Correlation is a powerful exploratory tool, but domain knowledge and additional analysis are required for causal inference and model building.


In [ ]:
# Demonstrate correlation limitations with examples
print('=' * 70)
print('⚠️ DEMONSTRATING CORRELATION LIMITATIONS')
print('=' * 70)

# 1. Non-linear relationship example
x = np.linspace(-3, 3, 100)
y_quadratic = x**2 + np.random.normal(0, 0.5, 100)
corr_quad = np.corrcoef(x, y_quadratic)[0, 1]

print(f'\n1️⃣ Non-Linear Relationship Example (Quadratic):')
print(f'   Pearson correlation: {corr_quad:.3f} (appears uncorrelated!)')
print(f'   But relationship is perfectly deterministic: Y = X² + noise')

# 2. Outlier impact demonstration
x_clean = np.random.normal(0, 1, 98)
y_clean = np.random.normal(0, 1, 98)
x_with_outlier = np.append(x_clean, [10])
y_with_outlier = np.append(y_clean, [10])

corr_without = np.corrcoef(x_clean, y_clean)[0, 1]
corr_with = np.corrcoef(x_with_outlier, y_with_outlier)[0, 1]

print(f'\n2️⃣ Outlier Impact Example:')
print(f'   Correlation without outlier: {corr_without:.3f}')
print(f'   Correlation with single outlier: {corr_with:.3f}')
print(f'   One point changed correlation by {abs(corr_with - corr_without):.3f}!')

# 3. Compare Pearson vs Spearman in our dataset
print(f'\n3️⃣ Pearson vs Spearman Comparison (detecting non-linearity):')
non_linear_candidates = []
for col in df.columns:
    if col != 'price':
        pearson_r = pearson_corr.loc[col, 'price']
        spearman_r = spearman_corr.loc[col, 'price']
        diff = abs(abs(pearson_r) - abs(spearman_r))
        if diff > 0.1:  # Significant difference
            non_linear_candidates.append((col, pearson_r, spearman_r, diff))

if non_linear_candidates:
    print(f'   Features with non-linear relationship to price (|Pearson - Spearman| > 0.1):')
    for feat, p, s, d in sorted(non_linear_candidates, key=lambda x: x[3], reverse=True):
        print(f'      • {feat}: Pearson={p:.3f}, Spearman={s:.3f}, Diff={d:.3f}')
else:
    print(f'   No strong non-linear relationships detected in this dataset.')

print(f'\n💡 Key Insight: When Pearson << Spearman, the relationship is monotonic but non-linear!')


## 🛠️ Hands-On Exercises

Apply your correlation analysis skills with these progressive exercises!

### Exercise 1: Basic Correlation Computation
Compute the Pearson correlation matrix for the dataset and display the correlation between `living_area` and `price`. Is this relationship strong or weak?


In [ ]:
# Exercise 1: Your code here



### Exercise 2: Correlation Method Comparison
Calculate and compare Pearson, Spearman, and Kendall correlations between `age_years` and `price`. Which method shows the strongest relationship? What does this tell you about the relationship type?


In [ ]:
# Exercise 2: Your code here



### Exercise 3: Styled Heatmap Creation
Create a correlation heatmap with the following customizations: (1) use a 'coolwarm' colormap, (2) mask the upper triangle, (3) add a title, (4) set figure size to (10, 8), and (5) rotate x-axis labels 45 degrees.


In [ ]:
# Exercise 3: Your code here



### Exercise 4: Target Correlation Ranking
Create a horizontal bar plot showing the top 5 features most correlated with `price` (by absolute value). Color positive correlations green and negative correlations red. Add value labels on each bar.


In [ ]:
# Exercise 4: Your code here



### Exercise 5: Multicollinearity Detection
Identify all feature pairs with correlation > 0.75. For each pair, recommend which feature to drop based on which has lower correlation with the target variable `price`.


In [ ]:
# Exercise 5: Your code here



### Exercise 6: Scatter Plot Analysis
Create a scatter plot of `square_feet` vs `living_area` with a regression line. Calculate and display the R-squared value in the title. Does this high correlation make sense from a domain perspective?


In [ ]:
# Exercise 6: Your code here



### Exercise 7: Feature Selection Implementation
Implement a function that takes a correlation matrix and returns a list of features to keep, removing any feature that has >0.8 correlation with another feature AND lower target correlation. Apply it to this dataset.


In [ ]:
# Exercise 7: Your code here



### Exercise 8: Correlation vs Causation Example
Create a synthetic example showing that two variables can be highly correlated without causation (e.g., both depend on a third variable). Demonstrate with a scatter plot and calculate the correlation.


In [ ]:
# Exercise 8: Your code here



### Exercise 9: VIF Calculation and Interpretation
Calculate VIF for all features in the dataset. Create a bar plot showing VIF values. Identify which features have VIF > 10 and explain why they might be problematic for linear regression.


In [ ]:
# Exercise 9: Your code here



### Exercise 10: Comprehensive Correlation Report
Write a function `generate_correlation_report(df, target)` that returns a dictionary containing: (1) top 5 positive correlations with target, (2) top 5 negative correlations, (3) high multicollinearity pairs, (4) recommended features to drop, and (5) final feature list. Test it on this dataset.


In [ ]:
# Exercise 10: Your code here



## Solutions & Key Insights (Review After Attempting) 🔑

<details>
<summary>Click to expand solutions after attempting the exercises</summary>

### Exercise 1 Solution
```python
corr_matrix = df.corr()
living_price_corr = corr_matrix.loc['living_area', 'price']
print(f'Correlation: {living_price_corr:.3f}')
# Expected: ~0.8 or higher (very strong, as living_area derived from square_feet)
```
**Insight**: Living area and square feet are highly correlated by construction—classic multicollinearity example.

### Exercise 2 Solution
```python
from scipy.stats import pearsonr, spearmanr, kendalltau
p_r, _ = pearsonr(df['age_years'], df['price'])
s_r, _ = spearmanr(df['age_years'], df['price'])
k_r, _ = kendalltau(df['age_years'], df['price'])
print(f'Pearson: {p_r:.3f}, Spearman: {s_r:.3f}, Kendall: {k_r:.3f}')
# Spearman often stronger for monotonic but non-linear relationships
```
**Insight**: If Spearman > Pearson, the relationship is monotonic but non-linear (common with age/price).

### Exercise 3 Solution
```python
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(df.corr(), dtype=bool), k=1)
sns.heatmap(df.corr(), mask=mask, cmap='coolwarm', center=0, 
            annot=True, fmt='.2f', ax=ax)
plt.xticks(rotation=45, ha='right')
plt.title('Custom Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
```
**Insight**: Custom styling improves readability for presentations and reports.

### Exercise 4 Solution
```python
top5 = df.corr()['price'].drop('price').abs().sort_values(ascending=False).head(5)
colors = ['green' if df.corr().loc[f, 'price'] > 0 else 'red' for f in top5.index]
plt.barh(top5.index, top5.values, color=colors)
for i, v in enumerate(top5.values):
    plt.text(v + 0.01, i, f'{v:.3f}', va='center')
plt.title('Top 5 Feature Correlations with Price')
plt.show()
```
**Insight**: Visual ranking immediately shows which features drive price predictions.

### Exercise 5 Solution
```python
high_corr = []
for i in range(len(df.columns)):
    for j in range(i+1, len(df.columns)):
        if abs(df.corr().iloc[i, j]) > 0.75:
            feat1, feat2 = df.columns[i], df.columns[j]
            if feat1 != 'price' and feat2 != 'price':
                corr1 = abs(df.corr().loc[feat1, 'price'])
                corr2 = abs(df.corr().loc[feat2, 'price'])
                drop = feat1 if corr1 < corr2 else feat2
                high_corr.append((feat1, feat2, drop))
```
**Insight**: Always keep the feature with stronger target correlation when removing multicollinearity.

### Exercise 6 Solution
```python
plt.scatter(df['square_feet'], df['living_area'], alpha=0.5)
z = np.polyfit(df['square_feet'], df['living_area'], 1)
p = np.poly1d(z)
plt.plot(df['square_feet'].sort_values(), p(df['square_feet'].sort_values()), 'r--')
r_squared = df.corr().loc['square_feet', 'living_area']**2
plt.title(f'Square Feet vs Living Area (R² = {r_squared:.3f})')
plt.xlabel('Square Feet')
plt.ylabel('Living Area')
plt.show()
# Domain insight: Living area is typically ~85% of square feet (garage, basement excluded)
```
**Insight**: High R² confirms these measure similar concepts—keep only one to avoid multicollinearity.

### Exercise 7 Solution
```python
def select_features(corr_matrix, target, threshold=0.8):
    target_corr = corr_matrix[target].abs().sort_values(ascending=False)
    features = list(target_corr.index)
    features.remove(target)
    
    keep = []
    for feat in features:
        redundant = any(abs(corr_matrix.loc[feat, k]) > threshold for k in keep)
        if not redundant:
            keep.append(feat)
    return keep

selected = select_features(df.corr(), 'price')
print(f'Selected {len(selected)} features: {selected}')
```
**Insight**: Greedy selection based on target correlation preserves most predictive features.

### Exercise 8 Solution
```python
np.random.seed(42)
temperature = np.random.normal(20, 10, 100)  # Confounding variable
ice_cream = temperature * 10 + np.random.normal(0, 20, 100)  # Depends on temp
drownings = temperature * 0.5 + np.random.normal(0, 2, 100)  # Also depends on temp

plt.scatter(ice_cream, drownings)
plt.title(f'Spurious Correlation: r = {np.corrcoef(ice_cream, drownings)[0,1]:.3f}')
plt.xlabel('Ice Cream Sales')
plt.ylabel('Drowning Incidents')
plt.show()
# Both depend on temperature, not on each other!
```
**Insight**: Always consider confounding variables—correlation without theory is dangerous.

### Exercise 9 Solution
```python
from statsmodels.stats.outliers_influence import variance_inflation_factor
X = df.drop('price', axis=1)
vif = pd.DataFrame()
vif['Feature'] = X.columns
vif['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif = vif.sort_values('VIF', ascending=False)

plt.barh(vif['Feature'], vif['VIF'], color=['red' if x > 10 else 'green' for x in vif['VIF']])
plt.axvline(x=10, color='red', linestyle='--')
plt.title('VIF by Feature')
plt.show()
# High VIF features are redundant with other features
```
**Insight**: VIF > 10 indicates severe multicollinearity; consider PCA or feature removal.

### Exercise 10 Solution
Your function should return a dictionary with:
- `top_positive`: 5 features with highest positive correlation
- `top_negative`: 5 features with highest negative correlation  
- `multicollinearity_pairs`: List of (feat1, feat2, corr) tuples
- `recommended_drops`: Features to remove
- `final_features`: Clean feature list

Test by calling `generate_correlation_report(df, 'price')` and verifying all keys present.

</details>
